# From Numerical Integration to Physics-Informed Neural Networks
## A Hands-On Laboratory: Chaotic Double Pendulum

**MLAB — Complex Systems School**

---

This laboratory follows the **natural evolution** of differential equation solvers.


**Part 1 — Lab (core methods)**

| Stage | Method | Key idea |
|-------|--------|----------|
| 1 | RK4 | Classical numerical integration |
| 2 | MLP | Learn from data |
| 3 | PINN | Embed physics into training |
| 4 | Parametric PINN (PPINN) | Learn a family of solutions |

This Lab end with a **comparison against the RK4 reference**, and the notebook closes with a **final comparison across all seven methods**.

> **Goal:** Understand *why* each new approach was developed, not just *how* it works.

## Setup — Imports and Paths

We import the helper scripts located in the `scripts/` directory.
Each script encapsulates one stage of the lab, keeping the notebook
focused on **concepts and math** rather than implementation details.

All scripts are plain importable modules — there is no command-line
entry point. Everything is meant to be run from this notebook.


In [43]:
import sys, os
sys.path.insert(0, 'scripts')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# --- Part 1 (Lab) scripts ---------------------------------------------------
from rk4_simulator import simulate, generate_dataset, double_pendulum_ode
from train_mlp     import build_mlp, train_mlp, save_mlp, load_mlp
from train_pinn    import build_mlp as build_pinn_arch, train_pinn, save_pinn, load_pinn, IC_DEFAULT
from train_ppinn   import build_ppinn, train_ppinn, save_ppinn, load_ppinn

# --- Metrics & visualization -------------------------------------------------
from metrics import (compute_rmse, compute_mae, compute_error_over_time,
                      measure_inference_time, compute_physics_residual,
                      generate_metrics_report, plot_trajectories,
                      plot_error_over_time, plot_phase_portrait,
                      plot_validation)
from simulation_viz import (animate_comparison, pendulum_cartesian,
                             plot_parameters, plot_parameters_all,
                             plot_training_curve, animate_pendulum,
                             plot_chaos_sensitivity)

tf.random.set_seed(42)
np.random.seed(42)

DATA_PATH  = 'data/rk4_trajectories.npz'
MODEL_DIR  = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs('data', exist_ok=True)

print("TensorFlow version:", tf.__version__)


---
# Simulation of the double pendulum: RK4, MLP, PINN, PPINN

The core lab: classical integration, a purely data-driven model, a
physics-informed model, and a parametric physics-informed model that
generalizes across initial conditions.


---
## Stage 1 — Classical Numerical Integration (RK4)

### The Double Pendulum

Two rigid rods connected at a pivot. The state is:

$$\mathbf{q} = [\theta_1,\, \omega_1,\, \theta_2,\, \omega_2]^\top$$

where $\theta_i$ are angles and $\omega_i = \dot\theta_i$ are angular velocities.

### Equations of Motion (Lagrangian Mechanics)

$$\dot\theta_1 = \omega_1, \qquad \dot\theta_2 = \omega_2$$

$$\dot\omega_1 = \frac{-g(2m_1+m_2)\sin\theta_1 - m_2 g\sin(\theta_1-2\theta_2)
- 2\sin\Delta\,m_2(\omega_2^2 L_2 + \omega_1^2 L_1\cos\Delta)}
{L_1(2m_1+m_2-m_2\cos 2\Delta)}$$

$$\dot\omega_2 = \frac{2\sin\Delta[\omega_1^2 L_1(m_1+m_2)+g(m_1+m_2)\cos\theta_1
+\omega_2^2 L_2 m_2\cos\Delta]}
{L_2(2m_1+m_2-m_2\cos 2\Delta)}$$

where $\Delta = \theta_1-\theta_2$.

### The RK4 Method

Given $\dot{\mathbf{q}} = f(t,\mathbf{q})$, one step of size $h$:

$$k_1 = f(t,\,\mathbf{q}),\quad
k_2 = f\!\left(t+\tfrac{h}{2},\,\mathbf{q}+\tfrac{h}{2}k_1\right),\quad
k_3 = f\!\left(t+\tfrac{h}{2},\,\mathbf{q}+\tfrac{h}{2}k_2\right),\quad
k_4 = f(t+h,\,\mathbf{q}+h\,k_3)$$

$$\mathbf{q}(t+h) = \mathbf{q}(t) + \frac{h}{6}(k_1+2k_2+2k_3+k_4)$$

> **RK4 is our ground truth.** It is 4th-order accurate: the local truncation
> error is $\mathcal{O}(h^5)$.

### Discussion Questions
1. Why is RK4 preferred over the simpler Euler method?
2. What happens to the prediction if you double the step size $h$?
3. Why does every new set of initial conditions require rerunning the solver?


In [44]:
# --- Generate the dataset (50 trajectories, T=10s, dt=0.01s) ----------------
import time

print("Generating RK4 dataset...")
t0 = time.time()
dataset = generate_dataset(
    n_trajectories=50,
    t_span=(0.0, 10.0),
    dt=0.01,
    seed=42,
    save_path=DATA_PATH,
)
t_rk4 = time.time() - t0

t_arr = dataset['t']
trajs = dataset['trajectories']
ics   = dataset['ics']

print(f"Done in {t_rk4:.1f}s")
print(f"Shape: {trajs.shape}  (n_traj, n_steps, 4)")


### Stage 1b — Custom RK4 Simulation

Modify the variables in the cell below and re-run to explore your own initial conditions.
— All angles in **degrees**, angular velocities in **rad/s**.


In [45]:
# --- Configurable RK4 parameters ---------------------------------------------
# ======= MODIFY THESE VALUES =======
RK4_th1_deg = 90.0   # theta1 initial angle (deg)
RK4_om1     = 0.0    # omega1 initial angular velocity (rad/s)
RK4_th2_deg = 90.0   # theta2 initial angle (deg)
RK4_om2     = 0.0    # omega2 initial angular velocity (rad/s)
RK4_L1      = 1.0    # Rod length L1 (m)
RK4_L2      = 1.0    # Rod length L2 (m)
RK4_m1      = 1.0    # Mass m1 (kg)
RK4_m2      = 1.0    # Mass m2 (kg)
RK4_g       = 9.81   # Gravity g (m/s^2)
RK4_T       = 10.0   # Total simulation time T (s)
RK4_dt      = 0.01   # Integration step dt (s)
# ====================================

ic_custom = np.array([np.radians(RK4_th1_deg), RK4_om1,
                       np.radians(RK4_th2_deg), RK4_om2])
t_cus, traj_cus = simulate(ic_custom, t_span=(0, RK4_T), dt=RK4_dt,
                            m1=RK4_m1, m2=RK4_m2, L1=RK4_L1, L2=RK4_L2, g=RK4_g)
print(f"Custom RK4: {len(t_cus)} steps | T={RK4_T}s  dt={RK4_dt}s")
print(f"IC: theta1={RK4_th1_deg} deg  omega1={RK4_om1}  theta2={RK4_th2_deg} deg  omega2={RK4_om2}")
print(f"Params: L1={RK4_L1}  L2={RK4_L2}  m1={RK4_m1}  m2={RK4_m2}  g={RK4_g}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle(f"Custom RK4  theta1_0={RK4_th1_deg}  theta2_0={RK4_th2_deg}  "
             f"L1={RK4_L1}  L2={RK4_L2}  m1={RK4_m1}  m2={RK4_m2}  g={RK4_g}",
             fontweight='bold')
axes[0].plot(t_cus, np.degrees(traj_cus[:,0]), color='#2196F3', lw=1.5, label=r'$\theta_1$')
axes[0].plot(t_cus, np.degrees(traj_cus[:,2]), color='#F44336', lw=1.5, label=r'$\theta_2$')
axes[0].set_ylabel('Angle (deg)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(t_cus, traj_cus[:,1], color='#2196F3', lw=1.5, label=r'$\omega_1$')
axes[1].plot(t_cus, traj_cus[:,3], color='#F44336', lw=1.5, label=r'$\omega_2$')
axes[1].set_ylabel('Ang. vel. (rad/s)'); axes[1].set_xlabel('Time (s)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [46]:
# --- Plot a single RK4 reference trajectory ----------------------------------
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle("RK4 Reference Trajectory (IC index 0)", fontsize=13, fontweight='bold')

for ax, idx, label in zip(axes, [0, 2], [r"$\theta_1$ (rad)", r"$\theta_2$ (rad)"]):
    ax.plot(t_arr, trajs[0, :, idx], color="#2196F3", linewidth=1.5)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


---
## Stage 2 — Data-Driven Learning (MLP)

### Motivation
RK4 must rerun for every new IC. Can we train a neural network
$f_\theta(t) \approx (\theta_1(t), \theta_2(t))$ that is **instantaneous at inference**?

### Architecture

$$t \;\xrightarrow{\text{Input}}\; \underbrace{[64]\to[64]\to[64]\to[64]}_{\tanh} \;\xrightarrow{\text{linear}}\; (\hat\theta_1, \hat\theta_2)$$

### Training Objective

$$\mathcal{L}_{\text{MLP}} = \frac{1}{N}\sum_{i=1}^{N}\left[(\hat\theta_1(t_i)-\theta_1^\text{RK4}(t_i))^2
+ (\hat\theta_2(t_i)-\theta_2^\text{RK4}(t_i))^2\right]$$

Pure **supervised learning** — the network has no knowledge of physics.

### Discussion Questions
1. The MLP fits the training data well. Does that mean it understands the physics?
2. What do you expect to happen if you query the MLP at $t > 10$ s (outside training)?
3. How much data is needed to train an acceptable model?


### MLP Training Hyperparameters

Modify the variables in the cell below before training.


In [47]:
# ======= MLP TRAINING PARAMETERS - MODIFY AS NEEDED =======
MLP_EPOCHS   = 5000   # maximum training epochs
MLP_LR       = 1e-3   # Adam learning rate
MLP_BATCH    = 64     # mini-batch size
MLP_PATIENCE = 200    # early-stopping patience
MLP_N_HIDDEN = 4      # number of hidden layers
MLP_N_UNITS  = 64     # neurons per hidden layer
MLP_TRAJ_IDX = 0      # RK4 trajectory index to use as training data
# ============================================================
print(f"MLP config: {MLP_N_HIDDEN} layers x {MLP_N_UNITS} units | "
      f"{MLP_EPOCHS} epochs | lr={MLP_LR} | traj={MLP_TRAJ_IDX}")


In [48]:
import time
data  = np.load(DATA_PATH)
t_arr = data['t'].astype(np.float32)
trajs = data['trajectories'].astype(np.float32)

# Use trajectory MLP_TRAJ_IDX as training data
y_train = trajs[MLP_TRAJ_IDX, :, [0, 2]]    # theta1, theta2
y_train = y_train.T
mlp_model = build_mlp(n_hidden=MLP_N_HIDDEN, n_units=MLP_N_UNITS)
mlp_model.summary()


In [49]:
t0 = time.time()
hist_mlp = train_mlp(
    mlp_model, t_arr, y_train,
    epochs=MLP_EPOCHS, lr=MLP_LR,
    batch_size=MLP_BATCH, patience=MLP_PATIENCE,
)
mlp_train_time = time.time() - t0
print(f"MLP training: {mlp_train_time:.1f}s | final loss: {hist_mlp.history['loss'][-1]:.4e}")
save_mlp(mlp_model, f"{MODEL_DIR}/mlp_model.keras")


In [50]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Train loss
axes[0].plot(hist_mlp.history['loss'], label='Train loss')
axes[0].set_yscale('log')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Train Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation loss
axes[1].plot(hist_mlp.history['val_loss'], label='Val loss')
axes[1].set_yscale('log')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('Validation Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [51]:
# --- Compare MLP vs RK4 -------------------------------------------------
t_inp = t_arr.reshape(-1, 1)
pred_mlp = mlp_model(t_inp, training=False).numpy()

trajectories = {
    "RK4": y_train,
    "MLP": pred_mlp,
}
fig = plot_trajectories(t_arr, trajectories)
plt.show()


---
## Stage 3 — Physics-Informed Learning (PINN)

### Motivation
The MLP may violate the governing ODEs — it learns a mapping, not the physics.
A **PINN** adds the ODE residual directly to the loss function.

### Composite Loss

$$\mathcal{L}_{\text{PINN}} = \lambda_d\,\mathcal{L}_{\text{data}}
+ \lambda_p\,\mathcal{L}_{\text{physics}}
+ \lambda_{ic}\,\mathcal{L}_{\text{IC}}$$

**Physics residual** — computed via automatic differentiation:

$$\mathcal{L}_{\text{physics}} = \frac{1}{N_c}\sum_{j=1}^{N_c}
\left[\left(\frac{d^2\hat\theta_1}{dt^2} - \dot\omega_1^{\text{ODE}}\right)^2
+\left(\frac{d^2\hat\theta_2}{dt^2} - \dot\omega_2^{\text{ODE}}\right)^2\right]$$

The derivatives $d\hat\theta/dt$ are computed with `tf.GradientTape`,
so no numerical differentiation is needed.

**Fixed training IC:** $\theta_1^0=\pi/3,\;\theta_2^0=\pi/4,\;\omega_1^0=\omega_2^0=0$

### Discussion Questions
1. Why is the IC loss needed even though we have data at $t=0$?
2. The PINN uses *collocation points* — what are they and why?
3. The PINN is still trained for one fixed IC. What limitation does this impose?


### PINN Training Hyperparameters

Modify the variables below before training. The physics loss uses automatic differentiation
via `tf.GradientTape` — no numerical differentiation needed.


In [10]:
# ======= PINN TRAINING PARAMETERS - MODIFY AS NEEDED =======
PINN_EPOCHS      = 10_000  # training epochs
PINN_LR          = 1e-3    # Adam learning rate
PINN_N_HIDDEN    = 4       # hidden layers
PINN_N_UNITS     = 64      # neurons per layer
PINN_TRAJ_IDX    = 0       # trajectory index for data loss
PINN_N_COLLOC    = 1000    # collocation points for physics residual
PINN_LAMBDA_DATA = 1.0     # weight for data loss
PINN_LAMBDA_PHYS = 1.0     # weight for physics residual
PINN_LAMBDA_IC   = 10.0    # weight for initial condition loss
PINN_LOG_EVERY   = 1000    # print progress every N epochs
# ============================================================
print(f"PINN config: {PINN_N_HIDDEN}x{PINN_N_UNITS} | {PINN_EPOCHS} epochs | "
      f"lambda_d={PINN_LAMBDA_DATA} lambda_p={PINN_LAMBDA_PHYS} lambda_ic={PINN_LAMBDA_IC}")


In [11]:
pinn_model = build_pinn_arch(n_hidden=PINN_N_HIDDEN, n_units=PINN_N_UNITS)
pinn_model.summary()


In [12]:
t0 = time.time()
hist_pinn = train_pinn(
    pinn_model, t_arr, y_train,
    ic=IC_DEFAULT,
    n_colloc=PINN_N_COLLOC,
    t_max=float(t_arr[-1]),
    epochs=PINN_EPOCHS,
    lr=PINN_LR,
    lambda_data=PINN_LAMBDA_DATA,
    lambda_phys=PINN_LAMBDA_PHYS,
    lambda_ic=PINN_LAMBDA_IC,
    log_every=PINN_LOG_EVERY,
)
pinn_train_time = time.time() - t0
print(f"PINN training: {pinn_train_time:.1f}s")
save_pinn(pinn_model, f"{MODEL_DIR}/pinn_model.keras")


In [13]:
# --- PINN training metrics: one function call replaces the manual plot ------
fig = plot_training_curve(hist_pinn, title="PINN Training Metrics")
plt.show()
print(f"Final total loss: {hist_pinn[-1]['loss_total']:.4e}")


In [14]:
pred_pinn = pinn_model(t_inp, training=False).numpy()

trajectories = {
    "RK4":  y_train,
    "MLP":  pred_mlp,
    "PINN": pred_pinn,
}
fig = plot_trajectories(t_arr, trajectories)
plt.show()
fig2 = plot_error_over_time(t_arr, trajectories)
plt.show()


### PINN Validation — Custom Initial Conditions

Test the trained PINN on a **different** initial condition.
Note: the PINN was trained with a **fixed** IC (`IC_DEFAULT`), so expect degraded performance here.
Modify `VAL_PINN_*` and re-run.


In [15]:
# ======= PINN VALIDATION PARAMETERS - MODIFY AS NEEDED =======
VAL_PINN_th1_deg = 90.0   # theta1_0 (deg)
VAL_PINN_om1     = 0.0    # omega1_0 (rad/s)
VAL_PINN_th2_deg = 90.0   # theta2_0 (deg)
VAL_PINN_om2     = 0.0    # omega2_0 (rad/s)
VAL_PINN_T       = 10.0   # validation time (s)
# ===============================================================

ic_val_pinn = np.array([np.radians(VAL_PINN_th1_deg), VAL_PINN_om1,
                         np.radians(VAL_PINN_th2_deg), VAL_PINN_om2])
_, traj_val_pinn = simulate(ic_val_pinn, t_span=(0, VAL_PINN_T), dt=0.01)
t_val_pinn = np.linspace(0, VAL_PINN_T, traj_val_pinn.shape[0]).astype(np.float32)
y_val_pinn = traj_val_pinn[:, [0, 2]].astype(np.float32)
pred_val_pinn = pinn_model(t_val_pinn.reshape(-1, 1), training=False).numpy()

ic_label = (f"theta1_0={VAL_PINN_th1_deg} deg  omega1_0={VAL_PINN_om1}  "
            f"theta2_0={VAL_PINN_th2_deg} deg  omega2_0={VAL_PINN_om2}")
fig = plot_validation(t_val_pinn, y_val_pinn, pred_val_pinn, "PINN", ic_label)
plt.show()


---
## Stage 4 — Parametric PINN (PPINN)

### Motivation
Every new IC requires retraining the PINN. The **Parametric PINN** solves this
by including the IC as additional inputs:

$$f_\theta\!\left(t,\;\theta_1^0,\;\omega_1^0,\;\theta_2^0,\;\omega_2^0\right)
\longrightarrow (\hat\theta_1(t),\hat\theta_2(t))$$

One trained model can predict for **any IC within the training distribution**
without retraining.

### Extended Architecture

$$[t,\theta_1^0,\omega_1^0,\theta_2^0,\omega_2^0]
\xrightarrow{5\text{ inputs}}
\underbrace{[64]\to[64]\to[64]\to[64]}_{\tanh}
\xrightarrow{\text{linear}}(\hat\theta_1, \hat\theta_2)$$

### IC Loss (now parameterized)

$$\mathcal{L}_{\text{IC}} = \frac{1}{B}\sum_{b=1}^{B}
\left[f_\theta(0, \mathbf{q}_0^b) - [\theta_1^{0,b}, \theta_2^{0,b}]\right]^2$$

### Discussion Questions
1. Why does adding the IC as an input allow generalization across many trajectories?
2. Would this approach work if the IC range was very wide? What would happen?
3. How does the total training cost compare to training one PINN per IC?


### PPINN Training Hyperparameters

The Parametric PINN is trained on **all** trajectories in the dataset simultaneously.
Modify the variables below before training.


In [16]:
# ======= PPINN TRAINING PARAMETERS - MODIFY AS NEEDED =======
PPINN_EPOCHS      = 20_000  # training epochs
PPINN_LR          = 1e-3    # Adam learning rate
PPINN_N_HIDDEN    = 4       # hidden layers
PPINN_N_UNITS     = 64      # neurons per layer
PPINN_N_COLLOC    = 200     # collocation points per trajectory
PPINN_BATCH_TRAJS = 8       # trajectories per mini-batch
PPINN_LAMBDA_DATA = 1.0     # weight for data loss
PPINN_LAMBDA_PHYS = 1.0     # weight for physics residual
PPINN_LAMBDA_IC   = 10.0    # weight for initial condition loss
PPINN_LOG_EVERY   = 2000    # print progress every N epochs
# ============================================================
print(f"PPINN config: {PPINN_N_HIDDEN}x{PPINN_N_UNITS} | {PPINN_EPOCHS} epochs | "
      f"batch_trajs={PPINN_BATCH_TRAJS} | lambda_ic={PPINN_LAMBDA_IC}")


In [17]:
ppinn_model = build_ppinn(n_hidden=PPINN_N_HIDDEN, n_units=PPINN_N_UNITS)
ppinn_model.summary()


In [18]:
dataset = {
    't':            data['t'].astype(np.float32),
    'trajectories': data['trajectories'].astype(np.float32),
    'ics':          data['ics'].astype(np.float32),
}

t0 = time.time()
hist_ppinn = train_ppinn(
    ppinn_model, dataset,
    n_colloc_per_traj=PPINN_N_COLLOC,
    epochs=PPINN_EPOCHS, lr=PPINN_LR,
    batch_trajs=PPINN_BATCH_TRAJS,
    lambda_data=PPINN_LAMBDA_DATA,
    lambda_phys=PPINN_LAMBDA_PHYS,
    lambda_ic=PPINN_LAMBDA_IC,
    log_every=PPINN_LOG_EVERY,
)
ppinn_train_time = time.time() - t0
print(f"Parametric PINN training: {ppinn_train_time:.1f}s")
save_ppinn(ppinn_model, f"{MODEL_DIR}/ppinn_model.keras")


In [19]:
fig = plot_training_curve(hist_ppinn, title="PPINN Training Metrics")
plt.show()
print(f"Final total loss: {hist_ppinn[-1]['loss_total']:.4e}")


In [20]:
# --- Test on a SEEN IC --------------------------------------------------
ic_test   = ics[0].astype(np.float32)
ic_rep    = np.tile(ic_test, (len(t_arr), 1))
ppinn_inp = np.concatenate([t_inp, ic_rep], axis=1)
pred_ppinn = ppinn_model(ppinn_inp, training=False).numpy()

# --- Test on an UNSEEN IC ------------------------------------------------
ic_unseen = np.array([0.9, 0.0, 0.7, 0.0], dtype=np.float32)
_, traj_unseen_rk4 = simulate(ic_unseen, t_span=(0, 10), dt=0.01)
y_unseen_rk4 = traj_unseen_rk4[:, [0, 2]].astype(np.float32)

ic_unseen_rep    = np.tile(ic_unseen, (len(t_arr), 1))
ppinn_unseen_inp = np.concatenate([t_inp, ic_unseen_rep], axis=1)
pred_ppinn_unseen = ppinn_model(ppinn_unseen_inp, training=False).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, y_ref, y_pred, title in zip(
    axes,
    [y_train, y_unseen_rk4],
    [pred_ppinn, pred_ppinn_unseen],
    ["Seen IC (training)", "Unseen IC (generalization test)"],
):
    ax.plot(t_arr, y_ref[:, 0],  color="#2196F3", label="RK4 theta1")
    ax.plot(t_arr, y_pred[:, 0], color="#FF9800", linestyle="--", label="PPINN theta1")
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Time (s)")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### PPINN Validation — Fully Configurable Initial Conditions

The PPINN accepts **any** IC as input — no retraining needed!
Set `VAL_PPINN_*`, run the cell, and observe how the model generalizes
to initial conditions it has never explicitly seen.


In [21]:
# ======= PPINN VALIDATION PARAMETERS - MODIFY AS NEEDED =======
VAL_PPINN_th1_deg = 90.0   # theta1_0 (deg)
VAL_PPINN_om1     = 0.0    # omega1_0 (rad/s)
VAL_PPINN_th2_deg = 90.0   # theta2_0 (deg)
VAL_PPINN_om2     = 0.0    # omega2_0 (rad/s)
VAL_PPINN_T       = 10.0   # validation time (s)
# ===============================================================

ic_val_pp = np.array([np.radians(VAL_PPINN_th1_deg), VAL_PPINN_om1,
                       np.radians(VAL_PPINN_th2_deg), VAL_PPINN_om2], dtype=np.float32)
_, traj_val_pp = simulate(ic_val_pp, t_span=(0, VAL_PPINN_T), dt=0.01)
t_val_pp   = np.linspace(0, VAL_PPINN_T, traj_val_pp.shape[0]).astype(np.float32)
y_val_pp   = traj_val_pp[:, [0, 2]].astype(np.float32)
ic_rep_pp  = np.tile(ic_val_pp, (len(t_val_pp), 1))
inp_val_pp = np.concatenate([t_val_pp.reshape(-1, 1), ic_rep_pp], axis=1)
pred_val_pp = ppinn_model(inp_val_pp, training=False).numpy()

ic_label = (f"theta1_0={VAL_PPINN_th1_deg} deg  omega1_0={VAL_PPINN_om1}  "
            f"theta2_0={VAL_PPINN_th2_deg} deg  omega2_0={VAL_PPINN_om2}")
fig = plot_validation(t_val_pp, y_val_pp, pred_val_pp, "PPINN", ic_label)
plt.show()


---
# Final Comparison — All Seven Methods

We now compare RK4, MLP, PINN, PPINN together using:
- **RMSE** and **MAE** (accuracy)
- **Inference time** (computational cost)
- **Physics residual** (physical consistency)
- **Trajectory, error, phase-space, and parameter-analysis plots**


In [ ]:
# Reload saved models so this section can also be run on its own
mlp_model   = load_mlp(f"{MODEL_DIR}/mlp_model.keras")
pinn_model  = load_pinn(f"{MODEL_DIR}/pinn_model.keras")
ppinn_model = load_ppinn(f"{MODEL_DIR}/ppinn_model.keras")

In [ ]:
pred_mlp   = mlp_model(t_inp, training=False).numpy()
pred_pinn  = pinn_model(t_inp, training=False).numpy()
pred_ppinn = ppinn_model(ppinn_inp, training=False).numpy()

trajectories_all = {
    "RK4":   y_train,
    "MLP":   pred_mlp,
    "PINN":  pred_pinn,
    "PPINN": pred_ppinn,
}

models_dict = {
    "RK4":   None,
    "MLP":   mlp_model,
    "PINN":  pinn_model,
    "PPINN": ppinn_model,
}

training_times = {
    "RK4":   t_rk4,
    "MLP":   mlp_train_time,
    "PINN":  pinn_train_time,
    "PPINN": ppinn_train_time,
}

df_metrics = generate_metrics_report(
    models=models_dict,
    t=t_arr,
    trajectories=trajectories_all,
    training_times=training_times,
    is_ppinn={"PPINN": True},
    ics={"PPINN": ic_test},
    save_path="metrics_report.csv",
)
df_metrics


In [39]:
# --- Trajectories, error over time, and phase portrait for all methods -------
fig1 = plot_trajectories(t_arr, trajectories_all)
plt.show()

fig2 = plot_error_over_time(t_arr, trajectories_all)
plt.show()

fig3 = plot_phase_portrait(trajectories_all, angle_idx=0)
plt.show()


## Pendulum Simulation & Animation

### Parameter Analysis
Six-panel plot per model: angular positions, velocities, energy conservation,
phase portraits, and the chaotic trajectory of mass 2.

### Real-time Animation
Three-panel animation: physical system (with trajectory tail) + live theta1(t) + theta2(t).
The PPINN prediction is overlaid as a dashed line so you can compare against RK4.

### Chaos Sensitivity
Three trajectories with an initial condition difference of only 0.01 rad (~0.6 deg)
diverge exponentially — the hallmark of chaos.


In [ ]:
# --- Parameter analysis: one 6-panel figure per model -----------------------
# traj0 is (N, 4): [theta1, omega1, theta2, omega2] -- RK4 reference
traj0 = trajs[0]

# Predictions dict: {model_name: (N, 2) array of [theta1_pred, theta2_pred]}
predictions = {
    "MLP":   pred_mlp,
    "PINN":  pred_pinn,
    "PPINN": pred_ppinn,
}

# Generates one figure per model (RK4 + each NN), displays them sequentially
figs = plot_parameters_all(t_arr, traj0, predictions)
for fig in figs:
    plt.show()


In [41]:
# --- Real-time animation (interactive) ---------------------------------
# Works in Jupyter Lab/Notebook with either:
#   %matplotlib widget   (recommended -- live interactive window)
#   %matplotlib inline   (embedded HTML player via to_jshtml)

# Option A -- interactive widget (run this in a cell BEFORE the animation cell):
# %matplotlib widget

# Option B -- embedded HTML player (works everywhere, no extra setup):
from IPython.display import HTML

anim = animate_pendulum(
    t_arr, traj0,
    title="RK4 -- Reference (IC index 0)",
    nn_pred=pred_ppinn,
    nn_label="PPINN",
    speed=2,        # 2x playback speed (skip every 2nd frame)
    tail_len=150,   # trajectory tail length in frames
)
HTML(anim.to_jshtml())   # embedded player -- scroll to see controls


---
## Conclusions

After completing this laboratory, you should be able to state:

1. **RK4** remains the reference: accurate, deterministic, but must rerun per IC.
2. **MLP** offers fast inference but lacks physical consistency and generalizes poorly.
3. **PINN** embeds the governing equations into training, improving physical consistency
   and reducing data dependence — but is still IC-specific.
4. **Parametric PINN** learns a *family* of solutions, generalizing across ICs
   without retraining.

> The goal is not to eliminate prediction error, but to understand how
> each methodology manages this unavoidable limitation.
